# PCA Framework: Testing Hypotheses on VTA Dopamine & GABA Dynamics

This notebook implements a systematic analytical framework to test whether VTA neural population dynamics encode **reward value** (RPE theory) or **movement direction** (behavioral hypothesis).

## Hypotheses
- **H1**: Movement direction dominates latent structure (SpontFB PCA basis captures task data)
- **H2**: DA and GABA encode different latent variables
- **H3**: GABA trajectories reflect value at reward time
- **H4**: CS-evoked and CR-evoked phasic DA components are separable
- **H5**: Aversive stimuli test the value axis (FUTURE - awaiting airpuff data)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

from plot_pca_framework import (
    load_dataset,
    extract_neuron_data,
    extract_group_averaged_data,
    fit_pca,
    project_onto_pca,
    align_pca_signs,
    run_pca,
    slice_epoch,
    slice_window,
    smooth_trajectories,
    get_event_markers,
    analyze_dataset,
    cross_project,
    compute_trajectory_metrics,
    compute_reconstruction_r2,
    compute_subspace_overlap,
    compute_procrustes_distance,
    compute_cross_correlation,
    compute_cross_epoch_r2_matrix,
    plot_1d_pc_timecourses,
    build_overlay_figure,
    plot_scree_comparison,
    plot_speed_profiles,
    plot_cross_epoch_r2_matrix,
    plot_metric_comparison_table,
    EPOCHS,
)

logging.basicConfig(level=logging.WARNING)
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Configuration

In [ ]:
# Dataset definitions
DATASETS = {
    'SpontFB': {'mat_file': 'dataSpontFB.mat', 'var_name': 'dataSpontFB'},
    'CRFB':    {'mat_file': 'dataCRFB.mat',    'var_name': 'dataCRFB'},
    'ToneFB':  {'mat_file': 'dataToneFB.mat',  'var_name': 'dataToneFB'},
}

# Neuron groups
DA_GROUPS = ['DF', 'DB', 'D', 'DFB']
GABA_GROUPS = ['GF', 'GB', 'G', 'GFB']
ALL_GROUPS = DA_GROUPS + GABA_GROUPS

# Analysis parameters
N_COMPONENTS = 3
EVENT_IDX = 600
WINDOW = 150  # expanded from 120 to capture post-reward dynamics
DT = 0.01
SG_WINDOW = 11
SG_ORDER = 3

## Step 0: Load All Datasets & Verify DA NaN Fix

The DFB group has NaN at specific neurons (row 19 or 65) in CRFB and ToneFB. Our new `extract_neuron_data()` logs per-group statistics and proceeds after dropping those rows.

In [ ]:
# Run all 9 dataset x neuron-combo analyses
results = {}
combos = {'Dopamine': DA_GROUPS, 'GABA': GABA_GROUPS, 'Combined': ALL_GROUPS}

for ds_name, ds_cfg in DATASETS.items():
    for combo_name, groups in combos.items():
        key = f"{ds_name}_{combo_name}"
        try:
            r = analyze_dataset(
                mat_file=ds_cfg['mat_file'],
                var_name=ds_cfg['var_name'],
                dataset_name=ds_name,
                neuron_groups=groups,
                combo_label=combo_name,
                n_components=N_COMPONENTS,
                event_idx=EVENT_IDX,
                window=WINDOW,
                dt=DT,
                sg_window=SG_WINDOW,
                sg_order=SG_ORDER,
            )
            results[key] = r
            evr = r['explained_variance_ratio']
            print(f"OK  {key:30s}  n={r['n_neurons']:4d}  "
                  f"EVR=[{evr[0]:.3f}, {evr[1]:.3f}, {evr[2]:.3f}]  "
                  f"total={sum(evr):.3f}")
            # Print NaN stats
            for g, s in r['stats'].items():
                if s['dropped'] > 0:
                    print(f"     {g}: dropped {s['dropped']}/{s['total']} neurons (NaN)")
        except Exception as e:
            print(f"FAIL {key:30s}  {e}")

print(f"\nSuccessful: {len(results)}/9")

---
## H2: DA vs GABA Encode Different Latent Variables

Compare trajectory geometry between DA and GABA populations. Start with SpontFB (pure movement, no reward) where both populations work.

### H2.1: Scree Plot Comparison — DA vs GABA Eigenvalue Spectra

If GABA has steeper scree drop-off, it has more coordinated, lower-dimensional activity.

In [ ]:
# Fit PCA with more components to see full scree
scree_data = {}
for key in ['SpontFB_Dopamine', 'SpontFB_GABA', 'CRFB_GABA', 'ToneFB_GABA']:
    if key in results:
        r = results[key]
        n_comp = min(20, r['n_neurons'] - 1)
        pca_full = fit_pca(r['X'], n_comp)
        scree_data[key] = pca_full.explained_variance_ratio_

fig_scree = plot_scree_comparison(scree_data, title="Scree Plot: DA vs GABA")
plt.show()

### H2.2: Trajectory Metrics Comparison

Compare arc length, speed, curvature, and fwd-bwd separation across all analyses.

In [ ]:
# Collect metrics for all successful results
all_metrics = {k: v['metrics'] for k, v in results.items()}

fig_metrics = plot_metric_comparison_table(all_metrics)
plt.show()

# Print summary table
rows = []
for k, m in all_metrics.items():
    rows.append({
        'Analysis': k,
        'Fwd Arc Length': f"{m['fwd_arc_length']:.2f}",
        'Bwd Arc Length': f"{m['bwd_arc_length']:.2f}",
        'Mean Separation': f"{m['mean_separation']:.3f}",
        'Fwd Peak Speed': f"{np.max(m['fwd_speed']):.1f}",
        'Bwd Peak Speed': f"{np.max(m['bwd_speed']):.1f}",
    })
display(pd.DataFrame(rows))

---
## H3: GABA Trajectories Reflect Value at Reward Time

If GABA carries value info, trajectories should deflect at reward delivery.
- **ToneFB**: reward at event_idx + 100 (1s after tone)
- **SpontFB**: null control — no reward at that latency

### H3.1: 1D PC Timecourses — ToneFB GABA

Look for deflection at reward time on individual PCs.

In [ ]:
if 'ToneFB_GABA' in results:
    r = results['ToneFB_GABA']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='ToneFB GABA — 1D PC Timecourses (reward at +1.0s)',
        dataset_name='ToneFB',
    )
    plt.show()
else:
    print('ToneFB_GABA not available')

### H3.2: 1D PC Timecourses — SpontFB GABA (Null Control)

Same plot for SpontFB. No reward at +1.0s, so no deflection expected.

In [ ]:
if 'SpontFB_GABA' in results:
    r = results['SpontFB_GABA']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='SpontFB GABA — 1D PC Timecourses (null: no reward)',
        dataset_name='SpontFB',
    )
    plt.show()
else:
    print('SpontFB_GABA not available')

### H3.3: Speed Profiles — ToneFB vs SpontFB GABA

If value is encoded, expect a speed transient at reward time in ToneFB but not SpontFB.

In [ ]:
speed_metrics = {}
for key in ['ToneFB_GABA', 'SpontFB_GABA']:
    if key in results:
        speed_metrics[key] = results[key]['metrics']

if speed_metrics:
    fig_speed = plot_speed_profiles(speed_metrics, title='Speed Profiles: ToneFB vs SpontFB GABA')
    plt.show()

### H3.4: 1D PC Timecourses — ToneFB Dopamine

Same analysis for Dopamine. Compare whether DA shows reward-time deflection.

In [ ]:
if 'ToneFB_Dopamine' in results:
    r = results['ToneFB_Dopamine']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='ToneFB Dopamine — 1D PC Timecourses',
        dataset_name='ToneFB',
    )
    plt.show()
else:
    print('ToneFB_Dopamine not available (NaN issue not resolved?)')

---
## H4: CS-evoked vs CR-evoked Phasic DA Components Are Separable

ToneFB (aligned to tone/CS) and CRFB (aligned to CR/movement onset) have the **same neurons**. Direct cross-projection is valid.

### H4.1: Cross-Projection — CRFB ↔ ToneFB GABA (same neurons)

Fit PCA on ToneFB, project CRFB, and vice versa. Compare trajectory shapes.

In [ ]:
cross_results_h4 = {}

if 'ToneFB_GABA' in results and 'CRFB_GABA' in results:
    # Fit on ToneFB, project CRFB
    cross_results_h4['Tone→CR'] = cross_project(
        results['ToneFB_GABA'], results['CRFB_GABA'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    # Fit on CRFB, project ToneFB
    cross_results_h4['CR→Tone'] = cross_project(
        results['CRFB_GABA'], results['ToneFB_GABA'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )

    for name, cr in cross_results_h4.items():
        print(f"{name}:  R² = {cr['r2']:.4f}")

    # Subspace overlap
    overlap = compute_subspace_overlap(
        results['ToneFB_GABA']['pca'].components_,
        results['CRFB_GABA']['pca'].components_,
    )
    print(f"\nSubspace overlap (cos of principal angles): {overlap}")
else:
    print('ToneFB_GABA or CRFB_GABA not available')

### H4.2: Overlay — ToneFB vs CRFB GABA in ToneFB PC Space

In [ ]:
if 'Tone→CR' in cross_results_h4 and 'ToneFB_GABA' in results:
    tsets = [
        {
            'fwd_smooth': results['ToneFB_GABA']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['ToneFB_GABA']['smooth_data']['bwd_smooth'],
            'label': 'ToneFB (native)',
            'fwd_color': 'orangered',
            'bwd_color': 'royalblue',
            'dash': 'solid',
            'event_markers': results['ToneFB_GABA']['event_markers'],
        },
        {
            'fwd_smooth': cross_results_h4['Tone→CR']['smooth_data']['fwd_smooth'],
            'bwd_smooth': cross_results_h4['Tone→CR']['smooth_data']['bwd_smooth'],
            'label': 'CRFB (projected)',
            'fwd_color': 'darkorange',
            'bwd_color': 'steelblue',
            'dash': 'dash',
            'event_markers': cross_results_h4['Tone→CR']['event_markers'],
        },
    ]
    fig_overlay = build_overlay_figure(
        tsets, title='H4: ToneFB vs CRFB GABA (ToneFB PC space)'
    )
    fig_overlay.show()

### H4.3: Cross-Correlation of PC Timecourses (ToneFB vs CRFB)

If CS and CR bursts are the same event shifted in time, cross-correlation should peak at the reaction-time lag.

In [ ]:
if 'ToneFB_GABA' in results and 'CRFB_GABA' in results:
    fig, axes = plt.subplots(N_COMPONENTS, 2, figsize=(14, 3*N_COMPONENTS),
                             sharex=True)
    directions = ['fwd_smooth', 'bwd_smooth']
    dir_labels = ['Forward', 'Backward']

    for col, (direction, dlabel) in enumerate(zip(directions, dir_labels)):
        tone_smooth = results['ToneFB_GABA']['smooth_data'][direction]
        cr_smooth = results['CRFB_GABA']['smooth_data'][direction]

        for pc in range(N_COMPONENTS):
            lags, corr = compute_cross_correlation(
                tone_smooth[pc], cr_smooth[pc], dt=DT
            )
            axes[pc, col].plot(lags, corr, color='purple', linewidth=1.5)
            peak_lag = lags[np.argmax(corr)]
            axes[pc, col].axvline(peak_lag, color='red', linestyle='--',
                                  alpha=0.7, label=f'peak={peak_lag:.2f}s')
            axes[pc, col].axvline(0, color='grey', linestyle=':', alpha=0.5)
            axes[pc, col].set_ylabel(f'PC{pc+1}')
            axes[pc, col].legend(fontsize=8)
            axes[pc, col].grid(True, alpha=0.3)
            if pc == 0:
                axes[pc, col].set_title(f'{dlabel}', fontsize=12)

    axes[-1, 0].set_xlabel('Lag (s)')
    axes[-1, 1].set_xlabel('Lag (s)')
    fig.suptitle('H4: Cross-Correlation ToneFB vs CRFB GABA', fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

### H4.4: Same analysis for Dopamine (if NaN fix worked)

In [ ]:
cross_results_h4_da = {}

if 'ToneFB_Dopamine' in results and 'CRFB_Dopamine' in results:
    cross_results_h4_da['Tone→CR'] = cross_project(
        results['ToneFB_Dopamine'], results['CRFB_Dopamine'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    cross_results_h4_da['CR→Tone'] = cross_project(
        results['CRFB_Dopamine'], results['ToneFB_Dopamine'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    for name, cr in cross_results_h4_da.items():
        print(f"DA {name}:  R² = {cr['r2']:.4f}")

    overlap_da = compute_subspace_overlap(
        results['ToneFB_Dopamine']['pca'].components_,
        results['CRFB_Dopamine']['pca'].components_,
    )
    print(f"DA subspace overlap: {overlap_da}")

    # Overlay
    tsets = [
        {
            'fwd_smooth': results['ToneFB_Dopamine']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['ToneFB_Dopamine']['smooth_data']['bwd_smooth'],
            'label': 'ToneFB DA (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['ToneFB_Dopamine']['event_markers'],
        },
        {
            'fwd_smooth': cross_results_h4_da['Tone→CR']['smooth_data']['fwd_smooth'],
            'bwd_smooth': cross_results_h4_da['Tone→CR']['smooth_data']['bwd_smooth'],
            'label': 'CRFB DA (projected)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': cross_results_h4_da['Tone→CR']['event_markers'],
        },
    ]
    fig_overlay = build_overlay_figure(
        tsets, title='H4: ToneFB vs CRFB Dopamine (ToneFB PC space)'
    )
    fig_overlay.show()
else:
    print('ToneFB_Dopamine or CRFB_Dopamine not available')

---
## H1: Movement Direction Dominates Latent Structure

Test whether SpontFB PCA basis (pure movement, no reward) captures task-aligned data.

### H1.1: Direct Cross-Projection — CRFB ↔ ToneFB (same neurons)

Since CRFB and ToneFB share neurons, this is the cleanest cross-projection test.

In [ ]:
# Already computed in H4 — display R² and subspace overlap
print('=== GABA ===')
if cross_results_h4:
    for name, cr in cross_results_h4.items():
        print(f"  {name}:  R² = {cr['r2']:.4f}")

print('\n=== Dopamine ===')
if cross_results_h4_da:
    for name, cr in cross_results_h4_da.items():
        print(f"  {name}:  R² = {cr['r2']:.4f}")

### H1.2: Group-Averaged Cross-Projection — SpontFB ↔ Task Data

SpontFB has different neuron ordering, so we use group-level averages (mean per GF/GB/G/GFB) for projection.

In [ ]:
cross_results_h1 = {}

# GABA: SpontFB -> CRFB and ToneFB
for target_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if 'SpontFB_GABA' in results and target_name in results:
        key = f'Spont→{target_name.split("_")[0]}'
        cross_results_h1[key] = cross_project(
            results['SpontFB_GABA'], results[target_name],
            use_group_avg=True, neuron_groups=GABA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        print(f"GABA {key}: R² = {cross_results_h1[key]['r2']:.4f}")

# Reverse: Task -> SpontFB
for source_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if source_name in results and 'SpontFB_GABA' in results:
        key = f'{source_name.split("_")[0]}→Spont'
        cross_results_h1[key] = cross_project(
            results[source_name], results['SpontFB_GABA'],
            use_group_avg=True, neuron_groups=GABA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        print(f"GABA {key}: R² = {cross_results_h1[key]['r2']:.4f}")

### H1.3: Group-Averaged Cross-Projection — DA (SpontFB ↔ Task)

In [ ]:
for target_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if 'SpontFB_Dopamine' in results and target_name in results:
        key = f'DA Spont→{target_name.split("_")[0]}'
        cr = cross_project(
            results['SpontFB_Dopamine'], results[target_name],
            use_group_avg=True, neuron_groups=DA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        cross_results_h1[key] = cr
        print(f"{key}: R² = {cr['r2']:.4f}")

for source_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if source_name in results and 'SpontFB_Dopamine' in results:
        key = f'DA {source_name.split("_")[0]}→Spont'
        cr = cross_project(
            results[source_name], results['SpontFB_Dopamine'],
            use_group_avg=True, neuron_groups=DA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        cross_results_h1[key] = cr
        print(f"{key}: R² = {cr['r2']:.4f}")

### H1.4: Overlay — SpontFB vs Task Trajectories in SpontFB PC Space

In [ ]:
# GABA overlay: SpontFB native + CRFB + ToneFB projected
if 'SpontFB_GABA' in results:
    tsets = [
        {
            'fwd_smooth': results['SpontFB_GABA']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['SpontFB_GABA']['smooth_data']['bwd_smooth'],
            'label': 'SpontFB GABA (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['SpontFB_GABA']['event_markers'],
        },
    ]

    # Note: for group-averaged cross-projection, the PC space is different
    # (4-dimensional group-avg vs full neuron). We show the group-avg projections.
    for key, color_fwd, color_bwd in [
        ('Spont→CRFB', 'gold', 'teal'),
        ('Spont→ToneFB', 'lime', 'purple'),
    ]:
        if key in cross_results_h1:
            cr = cross_results_h1[key]
            tsets.append({
                'fwd_smooth': cr['smooth_data']['fwd_smooth'],
                'bwd_smooth': cr['smooth_data']['bwd_smooth'],
                'label': f'{cr["project_dataset"]} GABA (proj)',
                'fwd_color': color_fwd, 'bwd_color': color_bwd, 'dash': 'dash',
                'event_markers': cr['event_markers'],
            })

    if len(tsets) > 1:
        fig = build_overlay_figure(
            tsets,
            title='H1: SpontFB vs Task GABA (group-avg SpontFB PC space)',
        )
        fig.show()

---
## Cross-Epoch PCA: Systematic Epoch Comparison

For each dataset, fit PCA on one epoch and project another. The R² heatmap shows which epochs share latent structure.

In [ ]:
# Run cross-epoch analysis for each successful dataset x combo
for key, r in results.items():
    ds_name = r['config']['dataset_name']

    # Select epochs relevant to this dataset
    relevant_epochs = {}
    for ename, ecfg in EPOCHS.items():
        if ecfg['dataset'] == ds_name or ecfg['dataset'] == 'Any':
            # Check epoch is within data bounds
            if ecfg['end'] <= r['timesteps']:
                relevant_epochs[ename] = ecfg

    if len(relevant_epochs) < 2:
        continue

    print(f"\n=== {key} ===")
    print(f"Epochs: {list(relevant_epochs.keys())}")

    r2_df, pca_dict = compute_cross_epoch_r2_matrix(
        r['X'], r['timesteps'], relevant_epochs,
        neuron_groups_label=r['config']['combo_label'],
        n_components=N_COMPONENTS,
    )

    fig = plot_cross_epoch_r2_matrix(r2_df, title=f'Cross-Epoch R²: {key}')
    plt.show()
    display(r2_df.round(3))

---
## Summary: All Quantitative Metrics

In [ ]:
# Compile all results into a summary table
summary_rows = []

# Within-dataset metrics
for key, r in results.items():
    m = r['metrics']
    evr = r['explained_variance_ratio']
    summary_rows.append({
        'Analysis': key,
        'Type': 'Within-dataset',
        'N_neurons': r['n_neurons'],
        'PC1_var': f"{evr[0]:.3f}",
        'PC2_var': f"{evr[1]:.3f}",
        'PC3_var': f"{evr[2]:.3f}",
        'Total_var': f"{sum(evr):.3f}",
        'Fwd_arc': f"{m['fwd_arc_length']:.2f}",
        'Bwd_arc': f"{m['bwd_arc_length']:.2f}",
        'Mean_sep': f"{m['mean_separation']:.3f}",
        'R2': 'N/A',
    })

# Cross-projection metrics
for label, cr_dict in [('H4 GABA', cross_results_h4),
                        ('H4 DA', cross_results_h4_da),
                        ('H1', cross_results_h1)]:
    for name, cr in cr_dict.items():
        m = cr['metrics']
        summary_rows.append({
            'Analysis': f"{label}: {name}",
            'Type': 'Cross-projection',
            'N_neurons': '',
            'PC1_var': '', 'PC2_var': '', 'PC3_var': '', 'Total_var': '',
            'Fwd_arc': f"{m['fwd_arc_length']:.2f}",
            'Bwd_arc': f"{m['bwd_arc_length']:.2f}",
            'Mean_sep': f"{m['mean_separation']:.3f}",
            'R2': f"{cr['r2']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

---
## Interpretation Guide

### H1: Movement vs Value
- **R² > 0.7** for SpontFB → task projections → movement dominates
- **R² < 0.4** → task context/reward adds information beyond movement
- Compare GABA vs DA cross-projection R² values

### H2: DA vs GABA
- Scree plot: if GABA drops faster → more coordinated, lower-dimensional
- Metric differences: different arc lengths, separations, speeds → different encoding

### H3: GABA Value Encoding
- 1D PC timecourses: deflection at reward time (+1.0s) in ToneFB → value PC
- Speed transient at reward in ToneFB but NOT SpontFB → reward-specific
- If specific PC shows reward modulation → that PC encodes value

### H4: CS vs CR Separation
- Cross-correlation peak at reaction-time lag → same burst, time-shifted
- Different trajectory shapes → separable neural events
- Compare GABA and DA cross-correlation patterns

### H5: Aversive Stimuli (FUTURE)
- Awaiting airpuff data. Framework ready: fit on ToneFB, project airpuff.